# Task 3: Baseline model

In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers torch scikit-learn pandas requests

In [ ]:
from urllib import request
import requests
import pandas as pd
import logging
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import numpy as np

In [ ]:
# prepare logger
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# check gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Download Data

In [ ]:
for url, fname in [
    ("https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv", "dontpatronizeme_pcl.tsv"),
    ("https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv", "dev_semeval_parids-labels.csv"),
    ("https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv", "train_semeval_parids-labels.csv"),
]:
    r = requests.get(url)
    open(fname, "wb").write(r.content)
    print(f"Downloaded {fname}")

module_url = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py"
with request.urlopen(module_url) as f, open("dont_patronize_me.py", "w") as out:
    out.write(f.read().decode("utf-8"))
print("Downloaded dont_patronize_me.py")

# Fetch "Don't Patronize Me!" data manager module

In [ ]:
from dont_patronize_me import DontPatronizeMe

In [ ]:
dpm = DontPatronizeMe('.', '.')

In [ ]:
dpm.load_task1()

In [ ]:
# helper function to save predictions to an output file
def labels2file(p, outf_path):
    with open(outf_path,'w') as outf:
        for pi in p:
            outf.write(','.join([str(k) for k in pi])+'\n')

# Load paragraph IDs

In [ ]:
trids = pd.read_csv('train_semeval_parids-labels.csv')
teids = pd.read_csv('dev_semeval_parids-labels.csv')
trids.par_id = trids.par_id.astype(str)
teids.par_id = teids.par_id.astype(str)

# Rebuild training set

In [ ]:
data = dpm.train_task1_df

rows = []  # will contain par_id, label and text
for idx in range(len(trids)):
    parid = trids.par_id[idx]
    keyword = data.loc[data.par_id == parid].keyword.values[0]
    text = data.loc[data.par_id == parid].text.values[0]
    label = data.loc[data.par_id == parid].label.values[0]
    rows.append({
        'par_id': parid,
        'community': keyword,
        'text': text,
        'label': label
    })

trdf1 = pd.DataFrame(rows)
trdf1

# Rebuild test set

In [ ]:
rows = []  # will contain par_id, label and text
for idx in range(len(teids)):
    parid = teids.par_id[idx]
    keyword = data.loc[data.par_id == parid].keyword.values[0]
    text = data.loc[data.par_id == parid].text.values[0]
    label = data.loc[data.par_id == parid].label.values[0]
    rows.append({
        'par_id': parid,
        'community': keyword,
        'text': text,
        'label': label
    })

tedf1 = pd.DataFrame(rows)
print(f"Test set size: {len(tedf1)}")
tedf1

# Fine-tune facebook/roberta-base

In [ ]:
# Downsample negative instances (2:1 neg:pos ratio)
pcldf = trdf1[trdf1.label == 1]
npos = len(pcldf)
training_set1 = pd.concat([pcldf, trdf1[trdf1.label == 0][:npos * 2]]).reset_index(drop=True)
print(f"Training set size: {len(training_set1)} | PCL: {npos} | Non-PCL: {npos*2}")

In [ ]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
MODEL_NAME = 'FacebookAI/roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)
print('Model loaded:', MODEL_NAME)

In [ ]:
BATCH_SIZE = 16
MAX_LEN = 128
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

train_dataset = PCLDataset(
    texts=training_set1.text.tolist(),
    labels=training_set1.label.tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = PCLDataset(
    texts=tedf1.text.tolist(),
    labels=tedf1.label.tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader)


def eval_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)


print('Starting training...')
for epoch in range(NUM_EPOCHS):
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    preds, true_labels = eval_model(model, test_loader, device)
    f1 = f1_score(true_labels, preds, average='binary')
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Dev F1: {f1:.4f}")

## Predictions

In [ ]:
preds_task1, true_labels = eval_model(model, test_loader, device)
print(classification_report(true_labels, preds_task1, target_names=['Non-PCL', 'PCL']))
print('Prediction distribution:', Counter(preds_task1))

In [ ]:
labels2file([[k] for k in preds_task1], 'base.txt')
print('Predictions saved to base.txt')
!head -n 10 base.txt